In [ ]:
from pathlib import Path
import os


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "interface_analyzer" / "reproducibility").exists():
            return path
    raise RuntimeError("Could not find repository root containing interface_analyzer/reproducibility")


PROJECT_ROOT = find_repo_root()
REPRO_DIR = PROJECT_ROOT / "interface_analyzer" / "reproducibility"
DATASET_DIR = REPRO_DIR / "dataset"
LOCAL_OUTPUT_DIR = REPRO_DIR / "_local_outputs"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Full manuscript-scale post-processing files are intentionally not bundled.
# Set INTERFACE_ANALYZER_DATA to the directory containing those generated files.
FULL_DATA_ROOT = Path(os.environ.get("INTERFACE_ANALYZER_DATA", LOCAL_OUTPUT_DIR)).expanduser()
FULL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

# For quick local CFG tests this defaults to the bundled sample dataset.
CFG_DIR = Path(os.environ.get("INTERFACE_ANALYZER_CFG_DIR", DATASET_DIR)).expanduser()

print("Project root:", PROJECT_ROOT)
print("Bundled CFG dataset:", DATASET_DIR)
print("Analysis data root:", FULL_DATA_ROOT)
print("CFG input dir:", CFG_DIR)


In [ ]:
import os
print(os.getcwd())

In [ ]:
import re
from pathlib import Path
from typing import Dict, List, Union, Optional
import pandas as pd

def read_lammps_thermo_sections(
    log_file: Union[str, Path],
    *,
    coerce_numeric: bool = True,
    keep_empty: bool = False,
    key_style: str = "auto",
) -> Dict[str, pd.DataFrame]:
    """
    Parse a LAMMPS log file and return a dictionary of thermo DataFrames.

    A thermo section is defined as:
      - starts at a header line whose stripped text begins with: 'Step '
      - ends at the first line whose stripped text begins with: 'Loop time '

    The header line provides column names. All subsequent data rows up to
    (but not including) 'Loop time' are parsed into a DataFrame.

    Parameters
    ----------
    log_file : str | Path
        Path to the LAMMPS log file.
    coerce_numeric : bool, default True
        Convert columns to numeric where possible (non-numeric stay as object).
    keep_empty : bool, default False
        If True, keep sections that ended up with no valid rows (rare).
    key_style : {'auto','index','range'}, default 'auto'
        How to name the dictionary keys:
          - 'index': 'section_001', 'section_002', ...
          - 'range': 'sec001_Step=first->last'
          - 'auto' : uses 'range' when 'Step' exists, else 'index'.

    Returns
    -------
    Dict[str, pd.DataFrame]
        Mapping from a section key to its thermo DataFrame.

    Notes
    -----
    - Lines considered data must start with a number-like character
      (digits, '.', '+', '-'); the parser skips warnings and empty lines.
    - If a data row has a different number of fields than the header,
      it is skipped.
    """
    p = Path(log_file)
    if not p.exists():
        raise FileNotFoundError(f"Log file not found: {p}")

    # Read text (ignore undecodable bytes gracefully)
    with p.open("r", errors="ignore") as f:
        lines = f.readlines()

    sections: List[pd.DataFrame] = []
    headers: List[List[str]] = []

    in_block = False
    cols: Optional[List[str]] = None
    rows: List[List[str]] = []

    header_re = re.compile(r"^Step(\s+\S+)*\s*$")
    loop_re = re.compile(r"^Loop time\b")
    data_re = re.compile(r"^[\s]*[+\-0-9\.]")  # likely numeric start

    for line in lines:
        s = line.strip()

        # Start of a thermo block: header line
        if not in_block and header_re.match(s):
            cols = s.split()
            rows = []
            in_block = True
            continue

        # Inside a thermo block
        if in_block:
            # End of block
            if loop_re.match(s):
                # finalize current block
                if rows or keep_empty:
                    df = pd.DataFrame(rows, columns=cols) if rows else pd.DataFrame(columns=cols)
                    if coerce_numeric and not df.empty:
                        # Convert each column to numeric when possible
                        for c in df.columns:
                            df[c] = pd.to_numeric(df[c], errors="ignore")
                    sections.append(df)
                    headers.append(cols)
                # reset
                in_block = False
                cols = None
                rows = []
                continue

            # Likely data row
            if s and data_re.match(s):
                parts = s.split()
                # only accept rows matching header length
                if cols is not None and len(parts) == len(cols):
                    rows.append(parts)
                # else: skip silently (could be wrapped lines or other prints)
            # else: ignore non-data lines inside thermo block

    # Build key→DataFrame dictionary
    out: Dict[str, pd.DataFrame] = {}
    for i, df in enumerate(sections, start=1):
        if key_style == "index":
            key = f"section_{i:03d}"
        else:
            # 'auto' or 'range'
            if "Step" in df.columns and not df.empty:
                key = f"sec{i:03d}_Step={df['Step'].iloc[0]}->{df['Step'].iloc[-1]}"
            else:
                key = f"section_{i:03d}"
        out[key] = df.reset_index(drop=True)

    return out

In [ ]:
import matplotlib.pyplot as plt
from typing import List, Union, Optional
import pandas as pd

def plot_lammps_thermo(
    df: pd.DataFrame,
    x: str = "Step",
    y: Union[str, List[str]] = "Temp",
    *,
    title: Optional[str] = None,
    xlabel: Optional[str] = None,
    ylabel: Optional[str] = None,
    figsize: tuple = (7, 4),
    grid: bool = True,
    legend: bool = True,
    xlim: Optional[tuple] = None,
    ylim: Optional[tuple] = None,
    savepath: Optional[str] = None,
    show: bool = True,
):
    """
    Plot one or more LAMMPS thermo quantities vs a chosen X axis.
    Displays a single figure in Jupyter (avoids duplicate rendering).
    """
    if isinstance(y, str):
        y = [y]

    missing_cols = [col for col in [x] + y if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Columns not found in DataFrame: {missing_cols}")

    fig, ax = plt.subplots(figsize=figsize)

    for col in y:
        ax.plot(df[x], df[col], label=col, linewidth=1.5)

    ax.set_xlabel(xlabel or x)
    ax.set_ylabel(ylabel or (", ".join(y) if len(y) == 1 else "Values"))
    if title:
        ax.set_title(title)
    if grid:
        ax.grid(True, linestyle="--", alpha=0.6)
    if legend and len(y) > 1:
        ax.legend()
    if xlim:
        ax.set_xlim(*xlim)
    if ylim:
        ax.set_ylim(*ylim)

    plt.tight_layout()

    # --- revised behavior ---
    if savepath:
        fig.savefig(savepath, dpi=300)
        plt.close(fig)
    elif show:
        plt.show()
    else:
        plt.close(fig)

    # Only return figure when not shown (for further use)
    if not show or savepath:
        return fig


### Read logfile

In [ ]:
from pathlib import Path

# --- 1. 路径设置 ---
SAVE_DIR = FULL_DATA_ROOT / "thermo_logs"
logfile = SAVE_DIR / "100_010_longtime/log_repeat1.Al_100_010"
#logfile = "./log.Al_100_010"
thermo_sections = read_lammps_thermo_sections(logfile)
all_thermo = pd.concat(thermo_sections, names=["section", "row"])
all_thermo.index.get_level_values("section").unique()

### Average/STD of a property

In [ ]:
sec = "sec008_Step=1000000->6000000"
df9 = all_thermo.loc[sec].reset_index(drop=True)

# Mean / std of one column (e.g., Temp)
mean_T = df9["Temp"].mean()
std_T  = df9["Temp"].std()
mean_P = df9["Press"].mean()
std_P  = df9["Press"].std()
mean_Pxx = df9["Pxx"].mean()
std_Pxx  = df9["Pxx"].std()
mean_Pyy = df9["Pyy"].mean()
std_Pyy  = df9["Pyy"].std()
mean_Pzz = df9["Pzz"].mean()
std_Pzz  = df9["Pzz"].std()
mean_T, std_T, mean_P, std_P, mean_Pxx, std_Pxx, mean_Pyy, std_Pyy, mean_Pzz, std_Pzz 

In [ ]:
df9["Lx"].mean()

In [ ]:
df9_eq

### Quick plot several properties 

In [ ]:
plot_lammps_thermo(df9, y=["Pxx", "Pyy","Pzz"], title=sec)

### Smoothing

In [ ]:
df9["Temp_roll50"] = df9["Temp"].rolling(window=50, min_periods=1).mean()
plot_lammps_thermo(df9, y=["Temp", "Temp_roll50"], title="sec008 Temp (raw vs rolling)")

In [ ]:
df9["Pzz_roll50"] = df9["Pzz"].rolling(window=50, min_periods=1).mean()
plot_lammps_thermo(df9, y=["Pzz", "Pzz_roll50"], title="sec009 Pzz (raw vs rolling)")

In [ ]:
df9["Pxx_roll50"] = df9["Pxx"].rolling(window=50, min_periods=1).mean()
plot_lammps_thermo(df9, y=["Pxx", "Pxx_roll50"], title="sec009 Pxx (raw vs rolling)")

In [ ]:
df9["Pyy_roll50"] = df9["Pyy"].rolling(window=50, min_periods=1).mean()
plot_lammps_thermo(df9, y=["Pyy", "Pyy_roll50"], title="sec009 Pyy (raw vs rolling)")

### Combine two sections

In [ ]:
df7 = all_thermo.loc['sec007_Step=0->1000000'].reset_index(drop=True)
print(len(df7))
df8 = all_thermo.loc['sec008_Step=1000000->3000000'].reset_index(drop=True)
print(len(df8))
df78 = pd.concat([df7, df8], ignore_index=True).reset_index()
print(len(df78))
df78["step_relabel"] = df78.index * 1000
df78["Temp_roll100"] = df78["Temp"].rolling(window=100, min_periods=10).mean()
fig = plot_lammps_thermo(df78, x="step_relabel", y=["Temp", "Temp_roll100"], 
                         title="NPzH for 3ns")

In [ ]:
df78["Press_roll100"] = df78["Press"].rolling(window=100, min_periods=10).mean()
fig = plot_lammps_thermo(df78, x="step_relabel", y=["Press", "Press_roll100"], 
                         title="NPzH for 3ns")

### Select with quantity

In [ ]:
win = df9.query("Step >= 150000 and Step <= 200000")
print(win["Temp"].mean(), win["Press"].std())
plot_lammps_thermo(win, y=["Temp", "Press"], title=f"{sec} (150k–200k)")

In [ ]:
temp_wide = all_thermo["Temp"].unstack(level="section")  # rows: row index; cols: sections
temp_wide.head()

# Quick compare of means
temp_wide.mean()

In [ ]:
sec = "sec004_Step=30000->55000"
df4 = all_thermo.loc[sec].reset_index(drop=True)
plot_lammps_thermo(df4, y=["c_lt","c_st"])

# Multiple Y columns
#plot_lammps_thermo(df, y=["Temp", "Press", "Density"], title="Thermo evolution")